In [2]:
!pip install numpy tensorflow


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import TextVectorization, Embedding, SimpleRNN, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
import numpy as np

print("NumPy is working")
print(np.__version__)

NumPy is working
2.5.3


In [3]:
import tensorflow as tf

print("TensorFlow is working")
print(tf.__version__)

TensorFlow is working
2.21.0


In [4]:
# ------------------------------------------------------------
# 1. Set random seeds
# ------------------------------------------------------------

np.random.seed(42)
tf.random.set_seed(42)

In [5]:
# ------------------------------------------------------------
# 2. Create a small labeled document dataset
#    0 = Sports
#    1 = Technology
# ------------------------------------------------------------

documents = np.array([
    "The football team won the match",
    "The player scored a brilliant goal",
    "The cricket team won the tournament",
    "The batsman scored a century",
    "The coach announced the new team",
    "The tennis player won the championship",
    "The basketball team played very well",
    "The football match was exciting",
    "The cricket player hit six boundaries",
    "The team practiced before the match",
    "The sports tournament starts tomorrow",
    "The goalkeeper saved the penalty",

    "The computer uses a powerful processor",
    "Artificial intelligence is changing technology",
    "The new smartphone has a powerful camera",
    "Machine learning is used in many applications",
    "The software update improves security",
    "The company developed a new computer system",
    "Cloud computing provides online services",
    "The laptop has a fast processor",
    "Python is widely used for machine learning",
    "The database stores large amounts of data",
    "Artificial intelligence uses neural networks",
    "The new application uses advanced technology"
])

labels = np.array([
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1
])



In [6]:
# ------------------------------------------------------------
# 3. Split dataset into training and testing data
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    documents,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels
)

print("Number of training documents:", len(X_train))
print("Number of testing documents :", len(X_test))

Number of training documents: 18
Number of testing documents : 6


In [7]:
# ------------------------------------------------------------
# 4. Text Vectorization
# ------------------------------------------------------------

max_words = 1000
sequence_length = 20

vectorizer = TextVectorization(
    max_tokens=max_words,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Learn vocabulary only from training data
vectorizer.adapt(X_train)


In [8]:
# ------------------------------------------------------------
# 5. Build RNN model
# ------------------------------------------------------------

model = Sequential([
    vectorizer,

    Embedding(
        input_dim=max_words,
        output_dim=32,
        mask_zero=True
    ),

    SimpleRNN(
        32,
        activation="tanh"
    ),

    Dense(
        16,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])


In [9]:
# ------------------------------------------------------------
# 6. Compile the model
# ------------------------------------------------------------

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [10]:
# ------------------------------------------------------------
# 7. Display model architecture
# ------------------------------------------------------------

model.build(input_shape=(None,))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 20)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 20, 32)         │        32,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,625 (135.25 KB)

 Trainable params: 34,625 (135.25 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# ------------------------------------------------------------
# 8. Convert text data to TensorFlow string tensors
# ------------------------------------------------------------

X_train_tf = tf.constant(X_train.tolist(), dtype=tf.string)
X_test_tf = tf.constant(X_test.tolist(), dtype=tf.string)

y_train_tf = tf.constant(y_train, dtype=tf.int32)
y_test_tf = tf.constant(y_test, dtype=tf.int32)

# ------------------------------------------------------------
# 9. Train the RNN
# ------------------------------------------------------------

history = model.fit(
    X_train_tf,
    y_train_tf,
    epochs=20,
    batch_size=4,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.6429 - loss: 0.6803 - val_accuracy: 0.5000 - val_loss: 0.6972
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8571 - loss: 0.6457 - val_accuracy: 0.5000 - val_loss: 0.6984
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.6148 - val_accuracy: 0.5000 - val_loss: 0.7040
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.5782 - val_accuracy: 0.5000 - val_loss: 0.7139
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.5365 - val_accuracy: 0.5000 - val_loss: 0.7284
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9286 - loss: 0.4922 - val_accuracy: 0.5000 - val_loss: 0.7482
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.4439 - val_accuracy: 0.5000 - val_loss: 0.7779
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.3978 - val_accuracy: 0.5000 - val_loss: 0.8178
Epoch 9

In [13]:
# ------------------------------------------------------------
# 10. Evaluate the model
# ------------------------------------------------------------

loss, accuracy = model.evaluate(
    X_test_tf,
    y_test_tf,
    verbose=0
)

print("\n====================================")
print("MODEL EVALUATION")
print("====================================")
print("Test Loss     :", round(loss, 4))
print("Test Accuracy :", round(accuracy * 100, 2), "%")


MODEL EVALUATION
Test Loss     : 1.0486
Test Accuracy : 66.67 %


In [15]:
# ------------------------------------------------------------
# 12. Make predictions
# ------------------------------------------------------------

probabilities = model.predict(
    X_test_tf,
    verbose=0
)

predictions = (
    probabilities >= 0.5
).astype(int).flatten()

print("\nPredictions generated successfully.")


Predictions generated successfully.


In [16]:
# ------------------------------------------------------------
# 13. Calculate Accuracy
# ------------------------------------------------------------

accuracy_value = accuracy_score(
    y_test_tf.numpy(),
    predictions
)

print("\n====================================")
print("TEST ACCURACY")
print("====================================")
print(
    "Accuracy:",
    round(accuracy_value * 100, 2),
    "%"
)


TEST ACCURACY
Accuracy: 66.67 %


In [17]:
# ------------------------------------------------------------
# 14. Classification Report
# ------------------------------------------------------------

print("\n====================================")
print("CLASSIFICATION REPORT")
print("====================================")

print(
    classification_report(
        y_test_tf.numpy(),
        predictions,
        target_names=[
            "Sports",
            "Technology"
        ],
        zero_division=0
    )
)


CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Sports       0.60      1.00      0.75         3
  Technology       1.00      0.33      0.50         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6



In [18]:
# ------------------------------------------------------------
# 15. Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test_tf.numpy(),
    predictions
)

print("\n====================================")
print("CONFUSION MATRIX")
print("====================================")

print(cm)


CONFUSION MATRIX
[[3 0]
 [2 1]]


In [19]:
# ------------------------------------------------------------
# 16. Predict New Documents
# ------------------------------------------------------------

new_documents = [
    "The new computer processor is very powerful",
    "The football player scored two goals"
]

# Convert to TensorFlow string tensor
new_documents_tf = tf.constant(
    new_documents,
    dtype=tf.string
)

new_probabilities = model.predict(
    new_documents_tf,
    verbose=0
).flatten()

print("\n====================================")
print("NEW DOCUMENT PREDICTIONS")
print("====================================")

for document, probability in zip(
    new_documents,
    new_probabilities
):

    if probability >= 0.5:
        category = "Technology"
    else:
        category = "Sports"

    print("\nDocument:", document)
    print("Predicted Category:", category)
    print(
        "Probability:",
        round(float(probability), 4)
    )


NEW DOCUMENT PREDICTIONS

Document: The new computer processor is very powerful
Predicted Category: Technology
Probability: 0.7458

Document: The football player scored two goals
Predicted Category: Sports
Probability: 0.0123


In [20]:
# ------------------------------------------------------------
# 17. Final Results
# ------------------------------------------------------------

print("\n")
print("==============================================")
print("       RNN DOCUMENT CLASSIFICATION")
print("==============================================")

print(
    "Test Accuracy:",
    round(accuracy_value * 100, 2),
    "%"
)

print("\nClass Labels:")
print("0 = Sports")
print("1 = Technology")

print("\nConfusion Matrix:")
print(cm)

print("\nModel successfully completed.")



       RNN DOCUMENT CLASSIFICATION
Test Accuracy: 66.67 %

Class Labels:
0 = Sports
1 = Technology

Confusion Matrix:
[[3 0]
 [2 1]]

Model successfully completed.
